# Chapter 8: CI/CD for Production Agentic Systems

Classical CI/CD was designed for deterministic software. Agentic systems introduce
**non-determinism**: the same input may produce different but equally valid outputs.

The solution is a **6-stage pipeline** that combines structural, semantic, and
operational validation gates before any deployment.

**Concepts:**
- LLM-as-Judge evaluation (multi-dimensional quality scoring)
- The 6-stage pipeline
- Quality gate enforcement
- Observability with Azure Application Insights

In [ ]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath('../..'))
from core.evaluation.llm_judge import EvaluationReport, EvaluationResult, LLMJudge
print('Evaluation module ready.')

## 8.1 Four-Dimensional Quality Scoring

A single "is it correct?" question is insufficient for LLM output.
We need a **rubric** with independent dimensions:

| Dimension | Question |
|-----------|---------|
| **Relevance** | Does it address the actual task? |
| **Groundedness** | Are claims supported by provided context? |
| **Coherence** | Is the reasoning internally consistent? |
| **Fluency** | Is it well-written, clear, and professional? |

A response can be fluent but ungrounded. Evaluating only one dimension misses the others.

In [ ]:
# Manual evaluation result (simulating a judge LLM response)
result = EvaluationResult(
    task='Explain how circuit breakers prevent cascading failures in microservices',
    response='Circuit breakers monitor error rates and stop forwarding requests when a \
threshold is exceeded, giving downstream services time to recover.',
    relevance=0.95,
    groundedness=0.90,
    coherence=0.92,
    fluency=0.88,
    pass_threshold=0.7,
    reasoning='Response directly addresses the question with the correct mechanism.',
)

import json
print(json.dumps(result.to_dict(), indent=2))
print(f'\nOverall score: {result.overall_score:.3f}')
print(f'Quality gate passed: {result.passed}')

## 8.2 LLM-as-Judge Batch Evaluation

In CI, every PR triggers a batch evaluation against a **golden test set** —
curated task-response pairs that represent the expected quality bar.

In [ ]:
import asyncio, json

# Simulate judge LLM responses (no real API needed)
MOCK_SCORES = [
    '{"relevance": 0.95, "groundedness": 0.88, "coherence": 0.92, "fluency": 0.90, "reasoning": "Accurate and complete."}',
    '{"relevance": 0.72, "groundedness": 0.65, "coherence": 0.70, "fluency": 0.85, "reasoning": "Partially relevant, lacks context."}',
    '{"relevance": 0.88, "groundedness": 0.91, "coherence": 0.89, "fluency": 0.93, "reasoning": "Well-grounded with examples."}',
]
call_idx = [0]

async def mock_judge(prompt: str) -> str:
    resp = MOCK_SCORES[call_idx[0] % len(MOCK_SCORES)]
    call_idx[0] += 1
    return resp

judge = LLMJudge(judge_fn=mock_judge, pass_threshold=0.75)

golden_set = [
    {'task': 'Explain circuit breaker pattern', 'response': 'A circuit breaker monitors error rates...'},
    {'task': 'What is event sourcing?', 'response': 'Event sourcing stores state as a log...'},
    {'task': 'Describe the Reflexion pattern', 'response': 'Reflexion iteratively improves responses...'},
]

async def main():
    report = await judge.evaluate_batch(golden_set, model_under_test='gpt-4o', prompt_version='v2.3')
    print(report.summary())
    print()
    print(f'CI Quality gate (threshold 0.8): {"PASS ✓" if report.quality_gate_passed(0.8) else "FAIL ✗"}')

asyncio.run(main())

## 8.3 The 6-Stage CI/CD Pipeline

```
Stage 1: Static Analysis
  ├─ ruff / mypy → syntax, type safety
  └─ git-secrets → no credentials in code

Stage 2: Unit Tests
  └─ pytest → all synchronous + async logic

Stage 3: LLM Evaluation (non-deterministic gate)
  └─ LLMJudge.evaluate_batch() → quality_gate_passed() must be True

Stage 4: Integration Tests
  └─ Agent collaboration end-to-end (mock Azure services)

Stage 5: Container Build + Security Scan
  ├─ docker build → produces OCI image
  └─ trivy scan → no HIGH/CRITICAL CVEs

Stage 6: Deployment (Staging → Production)
  ├─ az containerapp update --image new-image
  └─ smoke tests pass → promote to prod
```

Stage 3 is the key innovation: **no deployment if quality degrades**, even if tests pass.

In [ ]:
import asyncio

# Simulate the 6-stage gate checks
async def run_pipeline():
    stages = [
        ('Stage 1: Static Analysis', True, None),
        ('Stage 2: Unit Tests', True, '27 passed in 0.42s'),
        ('Stage 3: LLM Evaluation', None, None),  # computed
        ('Stage 4: Integration Tests', True, '8 scenarios passed'),
        ('Stage 5: Container Build', True, 'trivy: 0 HIGH CVEs'),
        ('Stage 6: Staging Deploy', True, 'smoke tests: OK'),
    ]

    # Run LLM evaluation for stage 3
    call_idx[0] = 0  # reset mock
    report = await judge.evaluate_batch(golden_set, model_under_test='gpt-4o', prompt_version='v2.3')
    evaluation_passed = report.quality_gate_passed(0.75)

    print('Pipeline execution:')
    print('=' * 50)
    all_passed = True
    for name, passed, detail in stages:
        if name.startswith('Stage 3'):
            passed = evaluation_passed
            detail = f'pass_rate={report.pass_rate:.0%}'
        if not passed:
            all_passed = False
        icon = '✅' if passed else '❌'
        detail_str = f' ({detail})' if detail else ''
        print(f'  {icon} {name}{detail_str}')
        if not passed:
            print('  ⛔ Pipeline halted — quality gate failed.')
            break
    if all_passed:
        print('=' * 50)
        print('✅ All stages passed. Agent deployed to production.')

asyncio.run(run_pipeline())

## 8.4 Observability with Azure Application Insights

Production agent observability requires custom metrics beyond HTTP status codes:

```python
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

configure_azure_monitor(
    connection_string=os.environ['APPINSIGHTS_CONNECTION_STRING']
)

tracer = trace.get_tracer('agentic-microservices')

async def traced_agent_call(task: str) -> dict:
    with tracer.start_as_current_span('agent.execute') as span:
        span.set_attribute('agent.task', task[:100])
        span.set_attribute('agent.model', 'gpt-4o')
        result = await agent.execute(task, {})
        span.set_attribute('agent.quality_score', result.get('score', 0))
        return result
```

Track these custom dimensions in Application Insights to detect quality drift over time.